## 01 - Récupération des données ADEME & Enedis

In [7]:
import requests
import pandas as pd

# URL du schéma du dataset
SCHEMA_URL = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/schema"

# Récupérer la structure du dataset
response = requests.get(SCHEMA_URL)
response.raise_for_status()
schema = response.json()

# Extraire les noms de colonnes
columns = [field["key"] for field in schema]
print(f"{len(columns)} colonnes trouvées :\n")
print(columns)

229 colonnes trouvées :

['numero_dpe', 'date_derniere_modification_dpe', 'date_visite_diagnostiqueur', 'date_etablissement_dpe', 'date_reception_dpe', 'date_fin_validite_dpe', 'numero_dpe_remplace', 'numero_dpe_immeuble_associe', 'id_rnb', 'provenance_id_rnb', 'numero_rpls_logement', 'numero_immatriculation_copropriete', 'modele_dpe', 'version_dpe', 'methode_application_dpe', 'etiquette_dpe', 'etiquette_ges', 'type_batiment', 'annee_construction', 'periode_construction', 'type_installation_chauffage', 'type_installation_ecs', 'hauteur_sous_plafond', 'nombre_appartement', 'nombre_niveau_immeuble', 'nombre_niveau_logement', 'typologie_logement', 'appartement_non_visite', 'surface_habitable_immeuble', 'surface_habitable_logement', 'surface_tertiaire_immeuble', 'classe_inertie_batiment', 'classe_altitude', 'zone_climatique', 'adresse_ban', 'numero_voie_ban', 'nom_rue_ban', 'nom_commune_ban', 'code_postal_ban', 'code_insee_ban', 'code_departement_ban', 'code_region_ban', 'identifiant_ban',

### Colonnes utiles à extraire

In [13]:
colonnes_utiles = [
    # --- Identification & temporalité ---
    "numero_dpe",
    "date_reception_dpe",
    "date_fin_validite_dpe",
    "annee_construction",
    "periode_construction",
    "modele_dpe",
    "version_dpe",
    "methode_application_dpe",

    # --- Localisation ---
    "code_postal_ban",
    "nom_commune_ban",
    "code_departement_ban",
    "code_region_ban",
    "zone_climatique",
    "coordonnee_cartographique_x_ban",
    "coordonnee_cartographique_y_ban",

    # --- Caractéristiques du logement ---
    "type_batiment",
    "typologie_logement",
    "surface_habitable_logement",
    "hauteur_sous_plafond",
    "nombre_niveau_logement",
    "inertie_lourde",
    "isolation_toiture",
    "qualite_isolation_murs",
    "qualite_isolation_menuiseries",

    # --- Équipements énergétiques ---
    "type_energie_principale_chauffage",
    "type_installation_chauffage",
    "type_installation_ecs",
    "type_ventilation",
    "presence_production_pv",
    "surface_totale_capteurs_pv",

    # --- Consommations et émissions ---
    "conso_5_usages_ep",
    "conso_chauffage_ep",
    "conso_ecs_ep",
    "conso_refroidissement_ep",
    "emission_ges_5_usages",
    "emission_ges_chauffage",
    "cout_total_5_usages",

    # --- Cibles ---
    "etiquette_dpe",
    "etiquette_ges"
]

### Chargement des codes postaux du département

In [21]:
import requests
import pandas as pd
import concurrent.futures
import os
import time

# === CONFIG ===
BASE_URL = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
CODES_POSTAUX_FILE = "../data/adresses-69.csv"   
OUTPUT_FILE = "../data/df_adem_69.csv"

# Charger les codes postaux du département
code_postaux_df = pd.read_csv(CODES_POSTAUX_FILE, dtype=str, sep=None, engine='python')
code_postals = code_postaux_df["code_postal"].unique().tolist()
print(code_postals)

['69790', '69170', '69250', '69380', '69009', '69008', '69006', '69007', '69005', '69001', '69004', '69002', '69003', '69780', '69360', '69124', '69580', '69720', '69960', '69320', '69800', '69140', '69330', '69970', '69730', '69740', '69150', '69680', '69510', '69390', '69910', '69100', '69640', '69770', '69400', '69430', '69820', '69460', '69200', '69120', '69420', '69670', '69890', '69620', '69240', '69160', '69220', '69440', '69490', '69700', '69560', '69590', '69270', '69870', '69210', '69850', '69930', '69690', '69550', '69650', '69830', '69290', '69230', '69610', '69110', '69190', '69450', '69370', '69280', '69470', '69480', '69530', '69310', '69600', '69350', '69860', '69760', '69840', '69540', '69520', '69340', '69130', '69570', '69660', '69115', '69260', '69410', '69630', '69300', '69500', '69126']


## 02 - Récupération des données ADEME

In [22]:
# Colonnes à extraire
#listeColumn = "numero_dpe,code_postal_ban,etiquette_dpe,date_reception_dpe,coordonnee_cartographique_x_ban,coordonnee_cartographique_y_ban"
listeColumn = ",".join(colonnes_utiles)

# Supprimer ancien fichier s'il existe
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

all_results = []

def fetch_data_smart(code_postal, etiquette=None, start_date=None, end_date=None):
    results = []
    size = 1000
    page = 1

    # Construction du filtre q
    q_parts = [f"code_postal_ban:{code_postal}"]
    if etiquette:
        q_parts.append(f"etiquette_dpe:{etiquette}")
    if start_date and end_date:
        q_parts.append(f"date_reception_dpe:[{start_date} TO {end_date}]")
    q_filter = " AND ".join(q_parts)

    print(f"\n--- Début téléchargement : {q_filter} ---")

    # Première requête pour connaître le total
    params = {"page": 1, "size": size, "qs": q_filter, "select": listeColumn, "q_fields": "code_postal_ban,etiquette_dpe,date_reception_dpe"}
    response = requests.get(BASE_URL, params=params)
    if response.status_code != 200:
        print(f" Erreur {response.status_code} pour {q_filter}")
        return results

    data = response.json()
    total = data.get("total", 0)
    print(f"Total estimé pour {q_filter} : {total}")

    # Si total > 1000 et pas encore filtré par etiquette
    if total > 10000 and etiquette is None:
        print(f"Nombre trop élevé ({total}), découpage par étiquette...")
        etiquettes = ["A", "B", "C", "D", "E", "F", "G"]
        for etiq in etiquettes:
            results.extend(fetch_data_smart(code_postal, etiquette=etiq))
        return results

    # Si total > 1000 et déjà filtré par etiquette mais pas par date
    if total > 10000 and etiquette is not None and start_date is None:
        print(f"Nombre trop élevé ({total}) pour {code_postal} et étiquette {etiquette}, découpage par année...")
        for year in range(2021, 2025):  # exemple années 2021 à 2024
            year_start = f"{year}-01-01"
            year_end = f"{year}-12-31"
            results.extend(fetch_data_smart(code_postal, etiquette, year_start, year_end))
        return results

    # Sinon récupération normale par page
    while True:
        params["page"] = page
        response = requests.get(BASE_URL, params=params)
        if response.status_code != 200:
            print(f" Erreur {response.status_code} page {page} pour {q_filter}")
            break

        page_results = response.json().get("results", [])
        if not page_results:
            print(f" Aucun résultat à la page {page}, arrêt.")
            break

        results.extend(page_results)
        print(f"{q_filter} Page {page} : {len(page_results)} résultats — cumul : {len(results)}/{total}")

        if len(page_results) < size:
            print(f" Dernière page atteinte pour {q_filter}")
            break

        page += 1
        time.sleep(3)

    print(f"--- Fin téléchargement : {q_filter} — {len(results)} DPE récupérés ---\n")
    return results

all_results = []
for cp in code_postals:
    all_results.extend(fetch_data_smart(cp))

df = pd.DataFrame(all_results)
df.to_csv("../data/df_adem_69.csv", index=False, encoding='utf-8')
print(f"Export terminé : df_adem_69.csv ({len(df)} lignes)")


--- Début téléchargement : code_postal_ban:69790 ---
Total estimé pour code_postal_ban:69790 : 198
code_postal_ban:69790 Page 1 : 198 résultats — cumul : 198/198
 Dernière page atteinte pour code_postal_ban:69790
--- Fin téléchargement : code_postal_ban:69790 — 198 DPE récupérés ---


--- Début téléchargement : code_postal_ban:69170 ---
Total estimé pour code_postal_ban:69170 : 2913
code_postal_ban:69170 Page 1 : 1000 résultats — cumul : 1000/2913
code_postal_ban:69170 Page 2 : 1000 résultats — cumul : 2000/2913
code_postal_ban:69170 Page 3 : 913 résultats — cumul : 2913/2913
 Dernière page atteinte pour code_postal_ban:69170
--- Fin téléchargement : code_postal_ban:69170 — 2913 DPE récupérés ---


--- Début téléchargement : code_postal_ban:69250 ---
Total estimé pour code_postal_ban:69250 : 3200
code_postal_ban:69250 Page 1 : 1000 résultats — cumul : 1000/3200
code_postal_ban:69250 Page 2 : 1000 résultats — cumul : 2000/3200
code_postal_ban:69250 Page 3 : 1000 résultats — cumul : 300

## 03 -Récupération des données Enedis

In [ ]:
import requests, pandas as pd
from io import BytesIO

base = "https://data.enedis.fr/api/explore/v2.1/catalog/datasets/consommation-annuelle-residentielle-par-adresse"
params = {"where": "code_departement='69'"}

r = requests.get(f"{base}/exports/csv", params=params, timeout=120)
r.raise_for_status()

df_enedis = pd.read_csv(BytesIO(r.content), sep=";", encoding="utf-8", low_memory=False)
print(df_enedis.shape)  

df_enedis.to_csv('../data/df_enedis_69.csv',index=False) 

(165267, 19)
